In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install wandb --quiet

## Config

Set `REPO_ROOT` to the absolute path of the repo on your Google Drive. All other paths are relative to it.

In [ ]:
import os
import sys
import types

import yaml
import torch
import wandb

In [ ]:
REPO_ROOT = "/content/drive/MyDrive/RutishauserLab/AI4Sci/METHODS/DiffusionBANMap/"
CONFIG_FILE = "config/betavae_full_celeba.yaml"
RUN_NAME = "run1"
RUN_NOTES = ""
RESUME_CKPT_PATH = None

## Imports

In [ ]:
sys.path.insert(0, REPO_ROOT)

import importlib.metadata
if not hasattr(importlib.metadata, '_patched'):
    _original_version = importlib.metadata.version
    def _patched_version(name):
        try:
            return _original_version(name)
        except importlib.metadata.PackageNotFoundError:
            return 'unknown'
    importlib.metadata.version = _patched_version
    importlib.metadata._patched = True

from data_utils import build_dataloaders
from models.beta_vae import BetaVAE
from scripts.betavae_trainer import BetaVAETrainer
from utils import discover_device, count_model_params

## Load config & build model

In [ ]:
from utils import make_run_tag

config_path = os.path.join(REPO_ROOT, CONFIG_FILE)
with open(config_path) as f:
    cfg = types.SimpleNamespace(**yaml.safe_load(f))

cfg.output_dir = os.path.join(REPO_ROOT, f"{cfg.output_dir.lstrip('./')}_{make_run_tag(cfg)}")
if not os.path.isabs(cfg.data_dir):
    cfg.data_dir = os.path.join(REPO_ROOT, cfg.data_dir.lstrip("./"))

device = torch.device(discover_device())
print(f"Device: {device}")

model = BetaVAE(
    input_channels=cfg.input_channels,
    input_height=cfg.input_height,
    input_width=cfg.input_width,
    latent_dim=cfg.latent_dim,
    hidden_dim=cfg.hidden_dim,
    beta=cfg.beta,
    device=device,
)
count_model_params(model)

## Data

In [ ]:
train_dataloader, val_dataloader = build_dataloaders(cfg)

## Wandb

In [ ]:
wandb.login(key="...")

wandb.init(
    project="diffusion_ban_map",
    name=f"{cfg.experiment_name}_{RUN_NAME}",
    notes=RUN_NOTES,
    config=vars(cfg),
)

## Train

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

trainer = BetaVAETrainer(
    model=model,
    optimizer=optimizer,
    scheduler=None,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    config=cfg,
    device=device,
)
trainer.train(resume_ckpt_path=RESUME_CKPT_PATH)